# МАНГА

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


In [1]:
# Установка необходимых библиотек
!pip install ultralytics
!pip install scikit-learn
!pip install opencv-python-headless
!pip install matplotlib
!pip install pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.8 MB/s eta 0:00:00


In [2]:
import os
import zipfile
import random
import shutil
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# 1. Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 2. Разархивируем файлы
zip_path = '/content/drive/MyDrive/Labeling.zip'
extract_path = '/content/Labeling'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Архив распакован")
else:
    print("Файл Labeling.zip не найден в /content/drive/MyDrive/")

Архив распакован


In [5]:
# 3. Пути к данным
images_dir = os.path.join(extract_path, 'converted_images')
labels_dir = os.path.join(extract_path, 'labels')
dataset_dir = '/content/dataset'

if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

In [6]:
# 4. Копируем изображения и разметку в единую структуру
image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpeg', '.jpg', '.png'))]

for img_file in image_files:
    # Копируем изображение
    src_img = os.path.join(images_dir, img_file)
    dst_img = os.path.join(dataset_dir, img_file)
    shutil.copy2(src_img, dst_img)

    # Копируем соответствующий txt файл
    txt_file = os.path.splitext(img_file)[0] + '.txt'
    src_txt = os.path.join(labels_dir, txt_file)
    dst_txt = os.path.join(dataset_dir, txt_file)
    if os.path.exists(src_txt):
        shutil.copy2(src_txt, dst_txt)

print(f"Скопировано {len(image_files)} изображений и соответствующих txt файлов")

Скопировано 765 изображений и соответствующих txt файлов


In [7]:
# 5. Разбиваем на train и val (80/20)
all_files = [f for f in os.listdir(dataset_dir) if f.endswith(('.jpeg', '.jpg', '.png'))]
train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)

# Создаем папки для YOLO
for split in ['train', 'val']:
    os.makedirs(os.path.join(dataset_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dataset_dir, split, 'labels'), exist_ok=True)

# Перемещаем файлы
for img_file in train_files:
    shutil.move(os.path.join(dataset_dir, img_file),
                os.path.join(dataset_dir, 'train', 'images', img_file))
    txt_file = os.path.splitext(img_file)[0] + '.txt'
    if os.path.exists(os.path.join(dataset_dir, txt_file)):
        shutil.move(os.path.join(dataset_dir, txt_file),
                    os.path.join(dataset_dir, 'train', 'labels', txt_file))

for img_file in val_files:
    shutil.move(os.path.join(dataset_dir, img_file),
                os.path.join(dataset_dir, 'val', 'images', img_file))
    txt_file = os.path.splitext(img_file)[0] + '.txt'
    if os.path.exists(os.path.join(dataset_dir, txt_file)):
        shutil.move(os.path.join(dataset_dir, txt_file),
                    os.path.join(dataset_dir, 'val', 'labels', txt_file))

print(f"Обучающая выборка: {len(train_files)} изображений")
print(f"Валидационная выборка: {len(val_files)} изображений")

Обучающая выборка: 612 изображений
Валидационная выборка: 153 изображений


In [8]:
import os
from PIL import Image

images_dir = '/content/Labeling/converted_images'
labels_dir = '/content/Labeling/labels'

print("=" * 60)
print(" ДИАГНОСТИКА ДАННЫХ")
print("=" * 60)

# 1. Статистика
all_files = os.listdir(images_dir)
image_files = [f for f in all_files if f.lower().endswith(('.jpeg', '.jpg', '.png'))]
txt_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]

print(f"\n Статистика:")
print(f"  Всего файлов в images: {len(all_files)}")
print(f"  Из них изображений: {len(image_files)}")
print(f"  TXT файлов: {len(txt_files)}")
print(f"  Разница: {len(image_files) - len(txt_files)}")

# 2. Находим проблемные файлы
print(f"\n Проверка файлов:")
problem_files = []

for f in image_files:
    # Проверяем пару
    txt_name = os.path.splitext(f)[0] + '.txt'
    if txt_name not in txt_files:
        problem_files.append(('no_txt', f))

    # Проверяем целостность
    try:
        img_path = os.path.join(images_dir, f)
        with Image.open(img_path) as img:
            img.verify()
    except Exception as e:
        problem_files.append(('corrupted', f, str(e)))

# 3. Выводим результаты
if problem_files:
    print(f"\n Найдено {len(problem_files)} проблемных файлов:")
    for problem in problem_files:
        if problem[0] == 'no_txt':
            print(f"  - Без разметки: {problem[1]}")
        elif problem[0] == 'corrupted':
            print(f"  - Поврежден: {problem[1]} ({problem[2][:50]}...)")
else:
    print("   Все файлы в порядке!")

print("\n" + "=" * 60)

 ДИАГНОСТИКА ДАННЫХ

 Статистика:
  Всего файлов в images: 765
  Из них изображений: 765
  TXT файлов: 765
  Разница: 0

 Проверка файлов:
   Все файлы в порядке!



In [9]:
import os

images_dir = '/content/Labeling/converted_images'
labels_dir = '/content/Labeling/labels'

# Получаем имена файлов без расширений
image_names = set([os.path.splitext(f)[0] for f in os.listdir(images_dir)
                   if f.lower().endswith(('.jpeg', '.jpg', '.png'))])
txt_names = set([os.path.splitext(f)[0] for f in os.listdir(labels_dir)
                 if f.endswith('.txt')])

# Находим TXT без пары
txt_without_image = txt_names - image_names

print(f" Итог:")
print(f"  Изображений: {len(image_names)}")
print(f"  TXT файлов: {len(txt_names)}")
print(f"  TXT без пары: {len(txt_without_image)}")

if txt_without_image:
    print(f"\n Найден TXT файл без изображения:")
    for f in txt_without_image:
        txt_path = os.path.join(labels_dir, f + '.txt')
        size = os.path.getsize(txt_path)
        print(f"   {f}.txt (размер: {size} байт)")

        # Показываем содержимое файла
        with open(txt_path, 'r') as file:
            content = file.read().strip()
            if content:
                print(f"     Содержимое: {content[:100]}...")
            else:
                print(f"     Содержимое: (пустой файл)")
else:
    print("\n Все TXT файлы имеют пару!")

 Итог:
  Изображений: 765
  TXT файлов: 765
  TXT без пары: 0

 Все TXT файлы имеют пару!


In [10]:
import os

images_dir = '/content/Labeling/converted_images'

# Все файлы
all_files = os.listdir(images_dir)
print(f" Всего файлов: {len(all_files)}")

# Изображения
image_files = [f for f in all_files if f.lower().endswith(('.jpeg', '.jpg', '.png'))]
print(f"  Изображений: {len(image_files)}")

# Другие файлы
other_files = [f for f in all_files if not f.lower().endswith(('.jpeg', '.jpg', '.png'))]
print(f" Других файлов: {len(other_files)}")

if other_files:
    print(f"\n Найден файл, который не является изображением:")
    for f in other_files:
        print(f"  - {f}")
        # Проверяем расширение
        ext = os.path.splitext(f)[1]
        print(f"    Расширение: '{ext}'")

 Всего файлов: 765
  Изображений: 765
 Других файлов: 0


In [11]:
# 6. Создаем data.yaml
yaml_content = f"""
path: {dataset_dir}
train: train/images
val: val/images

nc: 2
names: ['bubbles', 'text']
"""

with open(os.path.join(dataset_dir, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print("data.yaml создан")

data.yaml создан


In [12]:
# 7. Проверяем GPU
import torch
print(f" CUDA доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" GPU память: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# 8. ОБУЧЕНИЕ НА GPU
model = YOLO('yolov8n.pt')

results = model.train(
    data=os.path.join(dataset_dir, 'data.yaml'),
    epochs=80,
    imgsz=640,
    batch=16,  # Увеличиваем batch для GPU
    device=0,  # Используем GPU (0 - первая видеокарта)
    workers=8,  # Больше рабочих процессов для GPU
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    project='yolo_runs',
    name='bubbles_text',
    exist_ok=True,
    patience=50,  # Ранняя остановка при отсутствии улучшений
    save=True,
    save_period=10,  # Сохранять чекпоинт каждые 10 эпох
    verbose=True
)

print("✅ Обучение завершено!")

 CUDA доступен: True
 GPU: Tesla T4
 GPU память: 14.56 GB
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bubbles_text, nbs=64, nms=False

In [13]:
import os
import shutil

# YOLO всегда сохраняет как best.pt
original_best_path = '/content/runs/detect/yolo_runs/bubbles_text/weights/best.pt'
original_last_path = '/content/runs/detect/yolo_runs/bubbles_text/weights/last.pt'

# Сохраняем в Drive
drive_best_model = '/content/drive/MyDrive/best_yolo_model_V.pt'
drive_last_model = '/content/drive/MyDrive/last_yolo_model_V.pt'

# Копируем и переименовываем лучшую модель
if os.path.exists(original_best_path):
    shutil.copy2(original_best_path, drive_best_model)
    print(f" Лучшая модель сохранена как: {drive_best_model}")
    print(f" Размер: {os.path.getsize(original_best_path) / (1024*1024):.2f} MB")
else:
    print(f" Модель не найдена: {original_best_path}")

# Копируем и переименовываем последнюю модель
if os.path.exists(original_last_path):
    shutil.copy2(original_last_path, drive_last_model)
    print(f" Последняя модель сохранена как: {drive_last_model}")
    print(f" Размер: {os.path.getsize(original_last_path) / (1024*1024):.2f} MB")

print("\n Модели сохранены !")

 Лучшая модель сохранена как: /content/drive/MyDrive/best_yolo_model_V.pt
 Размер: 5.96 MB
 Последняя модель сохранена как: /content/drive/MyDrive/last_yolo_model_V.pt
 Размер: 5.96 MB

 Модели сохранены !
